# 03 — Baseline en Ridge

De weekbaseline kopieert hetzelfde uur vorige week. Ridge leert een gewogen som; regularisatie beperkt grote gewichten. StandardScaler wordt binnen iedere trainfit geleerd.

## Bijdragen — Messi10

**Raoul Buffels:** dataverwerking en cleaning, EDA en hypothesen, features, modellen,
tuning, evaluatie en documentatie; Citi Bike-frontend en backend.
**Jorre Van Dyck:** hulp bij deployment op Render.
Codex hielp met implementatie, uitleg, tests en uitvoering. Ieder teamlid moet de
eigen keuzes mondeling kunnen uitleggen.

We bepalen de projectmap en laden de kleine tabellen. Grote bronbestanden blijven op schijf.

In [1]:
from pathlib import Path
import sys, json
import pandas as pd
from IPython.display import display, Markdown
locations=[Path.cwd(),Path.cwd().parent,Path.cwd()/'Citybike_full']
ROOT=next(p for p in locations if (p/'citibike').exists())
sys.path.insert(0,str(ROOT))
from citibike.config import REPORTS, DATA, MODELS
def report(name):
    return json.loads((REPORTS/name).read_text(encoding='utf-8'))
def table(name):
    display(pd.read_csv(REPORTS/name,dtype={'start_station_id':'string','end_station_id':'string','station_id':'string'}))


## Modelinvoer en preprocessing

We laden exact de output van notebook 02: kalenderwaarden en historische tellingen. De vijf chronologische folds hebben ieder 28 latere validatiedagen. Alleen training mag preprocessing leren; de eindtest wordt niet geladen.

In [2]:
from time import perf_counter
import gc
from citibike.model_data import load_model_data
from citibike.features import FEATURES, CALENDAR, SavedChronologicalCV
from citibike.modeling import OperationalRegressor, SeasonalNaive, metrics
from citibike.reproduction import save_model_reproduction
training, validation, prepared=load_model_data()
X_train, y_train=training[FEATURES],training.rides
X_validation, y_validation=validation[FEATURES],validation.rides
display(pd.DataFrame(prepared['folds']))
display(pd.DataFrame({'feature':FEATURES}))
print(f'Training: {len(training):,} rijen; validatie: {len(validation):,}. Geen testdata.')


,train_start,train_stop,validation_start,validation_stop,train_last_day,validation_first_day,validation_last_day
0,0,109512,109512,110184,2025-12-11,2025-12-12,2026-01-08
1,0,110184,110184,110856,2026-01-08,2026-01-09,2026-02-05
2,0,110856,110856,111528,2026-02-05,2026-02-06,2026-03-05
3,0,111528,111528,112200,2026-03-05,2026-03-06,2026-04-02
4,0,112200,112200,112872,2026-04-02,2026-04-03,2026-04-30


,feature
0,hour_sin
1,hour_cos
2,weekday_sin
3,weekday_cos
4,year_sin
5,year_cos
6,is_weekend
7,clock_hours
8,lag_1d
9,lag_2d


Training: 112,872 rijen; validatie: 1,464. Geen testdata.


We importeren de concrete estimator en preprocessing.

In [3]:
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

## seasonal_naive

Geen hyperparameters; dit is onze eenvoudige referentie. De wrapper gebruikt dezelfde clipping en DST-regels als deployment.

In [4]:
model=OperationalRegressor(SeasonalNaive())
display(model)

OperationalRegressor(estimator=SeasonalNaive())

Hier trainen we het concrete model op de voorbereide trainingsrijen. Preprocessing leert alleen uit training.

In [5]:
started=perf_counter()
fitted=model.fit(X_train,y_train)
fit_seconds=perf_counter()-started

Hier voorspellen en evalueren we. RMSE, MAE en R² worden voor training en validatie berekend en tegen de oorspronkelijke scores gecontroleerd. Het rapport wordt apart opgeslagen; test en modelkeuze blijven behouden.

In [6]:
train_prediction=fitted.predict(X_train)
validation_prediction=fitted.predict(X_validation)
result=save_model_reproduction('seasonal_naive',training,validation,train_prediction,validation_prediction,fit_seconds,prepared)
display(pd.DataFrame([result]))
del fitted,model
gc.collect()

,name,RMSE,MAE,R2,train_RMSE,train_MAE,train_R2,fit_seconds,reproduced_at,prepared_sha256,fold_fingerprint,test_used,model_selection_changed,original_metrics_equal
0,seasonal_naive,2826.16492,1510.127732,0.668076,1309.747668,677.011659,0.798089,0.003045,2026-10-14T15:08:36.197009+00:00,2be4c1fa94f1ee4657020c8912af03e6aadd7735d8ba0d...,59238f467d526664e56943f96d102928ede96e4e371fec...,False,False,True


0

## ridge

De bestaande alpha is 10; alle 14 features blijven behouden. De wrapper gebruikt dezelfde clipping en DST-regels als deployment.

In [7]:
model=OperationalRegressor(make_pipeline(StandardScaler(),Ridge(alpha=10)))
display(model)

OperationalRegressor(estimator=Pipeline(steps=[('standardscaler',
                                                StandardScaler()),
                                               ('ridge', Ridge(alpha=10))]))

Hier trainen we het concrete model op de voorbereide trainingsrijen. Preprocessing leert alleen uit training.

In [8]:
started=perf_counter()
fitted=model.fit(X_train,y_train)
fit_seconds=perf_counter()-started

Hier voorspellen en evalueren we. RMSE, MAE en R² worden voor training en validatie berekend en tegen de oorspronkelijke scores gecontroleerd. Het rapport wordt apart opgeslagen; test en modelkeuze blijven behouden.

In [9]:
train_prediction=fitted.predict(X_train)
validation_prediction=fitted.predict(X_validation)
result=save_model_reproduction('ridge',training,validation,train_prediction,validation_prediction,fit_seconds,prepared)
display(pd.DataFrame([result]))
del fitted,model
gc.collect()

,name,RMSE,MAE,R2,train_RMSE,train_MAE,train_R2,fit_seconds,reproduced_at,prepared_sha256,fold_fingerprint,test_used,model_selection_changed,original_metrics_equal
0,ridge,1982.440036,1248.058114,0.836678,1021.691347,578.390365,0.877136,0.045709,2026-10-14T15:08:36.321771+00:00,2be4c1fa94f1ee4657020c8912af03e6aadd7735d8ba0d...,59238f467d526664e56943f96d102928ede96e4e371fec...,False,False,True


31

## ridge_calendar

Deze bestaande ablatie gebruikt alleen kalenderfeatures. De wrapper gebruikt dezelfde clipping en DST-regels als deployment.

In [10]:
model=OperationalRegressor(make_pipeline(StandardScaler(),Ridge(alpha=10)),columns=CALENDAR)
display(model)

OperationalRegressor(columns=['hour_sin', 'hour_cos', 'weekday_sin',
                              'weekday_cos', 'year_sin', 'year_cos',
                              'is_weekend', 'clock_hours'],
                     estimator=Pipeline(steps=[('standardscaler',
                                                StandardScaler()),
                                               ('ridge', Ridge(alpha=10))]))

Hier trainen we het concrete model op de voorbereide trainingsrijen. Preprocessing leert alleen uit training.

In [11]:
started=perf_counter()
fitted=model.fit(X_train,y_train)
fit_seconds=perf_counter()-started

Hier voorspellen en evalueren we. RMSE, MAE en R² worden voor training en validatie berekend en tegen de oorspronkelijke scores gecontroleerd. Het rapport wordt apart opgeslagen; test en modelkeuze blijven behouden.

In [12]:
train_prediction=fitted.predict(X_train)
validation_prediction=fitted.predict(X_validation)
result=save_model_reproduction('ridge_calendar',training,validation,train_prediction,validation_prediction,fit_seconds,prepared)
display(pd.DataFrame([result]))
del fitted,model
gc.collect()

,name,RMSE,MAE,R2,train_RMSE,train_MAE,train_R2,fit_seconds,reproduced_at,prepared_sha256,fold_fingerprint,test_used,model_selection_changed,original_metrics_equal
0,ridge_calendar,5315.257689,3986.439396,-0.174067,2325.245735,1617.036124,0.363611,0.019236,2026-10-14T15:08:36.412431+00:00,2be4c1fa94f1ee4657020c8912af03e6aadd7735d8ba0d...,59238f467d526664e56943f96d102928ede96e4e371fec...,False,False,True


31

## ridge_no_14d

Deze bestaande ablatie laat de 14-dagenlag weg; het is geen nieuwe featurekeuze. De wrapper gebruikt dezelfde clipping en DST-regels als deployment.

In [13]:
model=OperationalRegressor(make_pipeline(StandardScaler(),Ridge(alpha=10)),columns=[c for c in FEATURES if c!='lag_14d'])
display(model)

OperationalRegressor(columns=['hour_sin', 'hour_cos', 'weekday_sin',
                              'weekday_cos', 'year_sin', 'year_cos',
                              'is_weekend', 'clock_hours', 'lag_1d', 'lag_2d',
                              'lag_7d', 'same_hour_mean_7d',
                              'previous_day_total'],
                     estimator=Pipeline(steps=[('standardscaler',
                                                StandardScaler()),
                                               ('ridge', Ridge(alpha=10))]))

Hier trainen we het concrete model op de voorbereide trainingsrijen. Preprocessing leert alleen uit training.

In [14]:
started=perf_counter()
fitted=model.fit(X_train,y_train)
fit_seconds=perf_counter()-started

Hier voorspellen en evalueren we. RMSE, MAE en R² worden voor training en validatie berekend en tegen de oorspronkelijke scores gecontroleerd. Het rapport wordt apart opgeslagen; test en modelkeuze blijven behouden.

In [15]:
train_prediction=fitted.predict(X_train)
validation_prediction=fitted.predict(X_validation)
result=save_model_reproduction('ridge_no_14d',training,validation,train_prediction,validation_prediction,fit_seconds,prepared)
display(pd.DataFrame([result]))
del fitted,model
gc.collect()

,name,RMSE,MAE,R2,train_RMSE,train_MAE,train_R2,fit_seconds,reproduced_at,prepared_sha256,fold_fingerprint,test_used,model_selection_changed,original_metrics_equal
0,ridge_no_14d,2174.787271,1376.786629,0.803448,1060.539908,606.544454,0.867615,0.034526,2026-10-14T15:08:36.526910+00:00,2be4c1fa94f1ee4657020c8912af03e6aadd7735d8ba0d...,59238f467d526664e56943f96d102928ede96e4e371fec...,False,False,True


31

## Chronologische CV en train/validatie

We behouden de oorspronkelijke vijf CV-metingen naast de opnieuw uitgevoerde vaste fit. Dit is reproductie; we starten geen nieuwe modelselectie na de test.

In [16]:
for row in report('model_comparison.json'):
    if row['name'] in ['seasonal_naive', 'ridge', 'ridge_calendar', 'ridge_no_14d']:
        display(Markdown('**'+row['name']+'**'))
        display(pd.DataFrame(row['cv']))

**seasonal_naive**

,fold,RMSE,MAE,R2,seconds
0,1,1979.271967,1259.720238,0.146485,0.029741
1,2,1797.007685,1250.418155,0.337800,0.031044
2,3,1474.318125,965.205357,0.285118,0.035261
3,4,2565.008427,1614.552083,0.457230,0.036382
4,5,2617.758436,1660.016369,0.626783,0.038523


**ridge**

,fold,RMSE,MAE,R2,seconds
0,1,1410.358708,884.905921,0.566630,0.121878
1,2,1346.493144,883.285060,0.628210,0.131189
2,3,1176.981053,814.124750,0.544393,0.136288
3,4,1984.786066,1254.158220,0.675013,0.129334
4,5,2052.954364,1291.675370,0.770458,0.153673


**ridge_calendar**

,fold,RMSE,MAE,R2,seconds
0,1,1674.535138,1107.631297,0.389074,0.112595
1,2,1937.044243,1318.133977,0.230571,0.057944
2,3,1395.444717,982.618884,0.359562,0.059053
3,4,3485.826604,2433.370994,-0.002421,0.058408
4,5,4458.309832,3182.851477,-0.082538,0.060284


**ridge_no_14d**

,fold,RMSE,MAE,R2,seconds
0,1,1377.005274,871.687706,0.586885,0.094437
1,2,1178.267580,845.175477,0.715307,0.096270
2,3,1176.106068,789.552901,0.545070,0.101209
3,4,1846.263113,1224.848383,0.718793,0.091279
4,5,2212.117597,1428.963653,0.733486,0.050353


## Conclusie en tuningkeuze

Ridge verbetert de baseline maar valideert minder goed dan de niet-lineaire modellen. Alpha=10 bleef vast. RF en LightGBM waren de twee kandidaten die op basis van trainings-CV verder getuned werden. Kalender-only mist recente intensiteit; we behouden ook deze minder geslaagde experimenten.